# 싱글턴 vs 멀티턴 호출 차이

이 노트북은 Gemini API를 셀 단위로 천천히 호출하도록 구성했습니다.

- 한 번에 전체 파일을 실행하지 않고, 필요한 셀만 실행합니다.
- API 호출이 발생하는 셀은 `[API 요청 n]` 형태로 요청 수를 보여줍니다.
- `429 RESOURCE_EXHAUSTED`가 나오면 잠시 기다린 뒤 이어서 실행합니다.
- 수업 중에는 싱글턴 2회, 멀티턴 2~3회만 실행해도 핵심 차이를 충분히 확인할 수 있습니다.

## 0. 준비

`.env` 파일에 `GEMINI_API_KEY=...` 값이 있어야 합니다.

모델을 바꾸고 싶으면 `.env`에 아래처럼 추가할 수 있습니다.

```text
GEMINI_MODEL=gemini-3.5-flash
GEMINI_REQUEST_DELAY_SECONDS=1.0
```

In [1]:
import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GEMINI_API_KEY=... 값을 넣어주세요.")

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
REQUEST_DELAY_SECONDS = float(os.getenv("GEMINI_REQUEST_DELAY_SECONDS", "1.0"))

client = genai.Client(api_key=api_key)
api_request_count = 0

print(f"MODEL = {MODEL}")
print(f"REQUEST_DELAY_SECONDS = {REQUEST_DELAY_SECONDS}")

MODEL = gemini-3.5-flash
REQUEST_DELAY_SECONDS = 1.0


In [2]:
def extract_retry_delay(exc: errors.ClientError) -> str | None:
    details = exc.details if isinstance(exc.details, dict) else {}
    error_details = details.get("details") or details.get("error", {}).get("details", [])

    for item in error_details:
        if item.get("@type", "").endswith("RetryInfo"):
            return item.get("retryDelay")
    return None


def print_quota_error(exc: errors.ClientError) -> None:
    print("\n[Gemini API 할당량 초과]")
    print(f"  모델: {MODEL}")
    print(f"  현재 노트북 세션 요청 수: {api_request_count}회")

    retry_delay = extract_retry_delay(exc)
    if retry_delay:
        print(f"  잠시 후 재시도 가능: 약 {retry_delay}")

    print("  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.")
    print("  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.")


def call_gemini(description: str, callback):
    global api_request_count
    api_request_count += 1
    print(f"[API 요청 {api_request_count}] {description}")

    try:
        result = callback()
        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(REQUEST_DELAY_SECONDS)
        return result
    except errors.ClientError as exc:
        if exc.code == 429:
            print_quota_error(exc)
            return None
        raise


def stream_gemini(description: str, callback) -> None:
    global api_request_count
    api_request_count += 1
    print(f"[API 요청 {api_request_count}] {description}")
    print("AI: ", end="", flush=True)

    try:
        for chunk in callback():
            if chunk.text:
                print(chunk.text, end="", flush=True)
        print()
        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(REQUEST_DELAY_SECONDS)
    except errors.ClientError as exc:
        if exc.code == 429:
            print_quota_error(exc)
            return
        raise


def show_response(label: str, response, limit: int = 400) -> None:
    if response is None:
        return

    text = (response.text or "").strip()
    print(f"\n{label}")
    print("-" * len(label))
    print(text[:limit])

    usage = getattr(response, "usage_metadata", None)
    if usage:
        prompt = getattr(usage, "prompt_token_count", None)
        output = getattr(usage, "candidates_token_count", None)
        total = getattr(usage, "total_token_count", None)
        print(f"\n토큰 사용량: 입력={prompt}, 출력={output}, 총={total}")

## 1. 싱글턴 호출

싱글턴 호출은 매 요청이 독립적입니다. 다음 두 셀은 각각 별도의 요청이므로, 두 번째 질문의 `방금 말한 것`을 모델이 안정적으로 기억한다고 볼 수 없습니다.

In [3]:
question = "파이썬이 뭐야? 한 문장으로 답해줘."

response = call_gemini(
    "싱글턴 1: 첫 질문",
    lambda: client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(max_output_tokens=120),
    ),
)

print(f"Q: {question}")
show_response("A", response)

[API 요청 1] 싱글턴 1: 첫 질문

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 1회
  잠시 후 재시도 가능: 약 45s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
Q: 파이썬이 뭐야? 한 문장으로 답해줘.


In [4]:
question = "방금 말한 것보다 더 쉽게 설명해줘."

response = call_gemini(
    "싱글턴 2: 문맥 없는 후속 질문",
    lambda: client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(max_output_tokens=120),
    ),
)

print(f"Q: {question}")
show_response("A", response)

[API 요청 2] 싱글턴 2: 문맥 없는 후속 질문

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 2회
  잠시 후 재시도 가능: 약 39s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
Q: 방금 말한 것보다 더 쉽게 설명해줘.


## 2. 멀티턴 호출

`client.chats.create()`로 만든 `chat` 객체는 대화 기록을 보관합니다. 아래 셀들은 같은 `chat` 객체를 공유하므로 후속 질문이 앞 대화의 영향을 받습니다.

In [5]:
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(max_output_tokens=160),
)

print("멀티턴 chat 세션을 만들었습니다. 이 셀은 API 요청을 보내지 않습니다.")

멀티턴 chat 세션을 만들었습니다. 이 셀은 API 요청을 보내지 않습니다.


In [6]:
message = "파이썬이 뭐야? 한 문장으로 답해줘."

response = call_gemini(
    "멀티턴 1: 첫 메시지",
    lambda: chat.send_message(message),
)

print(f"나: {message}")
show_response("AI", response)

[API 요청 3] 멀티턴 1: 첫 메시지
나: 파이썬이 뭐야? 한 문장으로 답해줘.

AI
--
파이썬은 문

토큰 사용량: 입력=15, 출력=5, 총=171


In [7]:
message = "방금 말한 것보다 더 쉽게 설명해줘."

response = call_gemini(
    "멀티턴 2: 앞 대화를 참조하는 메시지",
    lambda: chat.send_message(message),
)

print(f"나: {message}")
show_response("AI", response)

[API 요청 4] 멀티턴 2: 앞 대화를 참조하는 메시지
나: 방금 말한 것보다 더 쉽게 설명해줘.

AI
--
파이썬은 컴퓨터가 일

토큰 사용량: 입력=34, 출력=7, 총=190


In [8]:
message = "예시 코드도 짧게 줘."

response = call_gemini(
    "멀티턴 3: 같은 문맥에서 코드 요청",
    lambda: chat.send_message(message),
)

print(f"나: {message}")
show_response("AI", response)

[API 요청 5] 멀티턴 3: 같은 문맥에서 코드 요청
나: 예시 코드도 짧게 줘.

AI
--
What is the simplest Python code?

토큰 사용량: 입력=52, 출력=7, 총=208


In [9]:
print("--- 현재 chat 대화 기록 ---")
for turn in chat.get_history():
    role = "나" if turn.role == "user" else "AI"
    text = turn.parts[0].text.strip().replace("\n", " ")
    print(f"[{role}] {text[:100]}")

--- 현재 chat 대화 기록 ---
[나] 파이썬이 뭐야? 한 문장으로 답해줘.
[AI] 파이썬은 문
[나] 방금 말한 것보다 더 쉽게 설명해줘.
[AI] 파이썬은 컴퓨터가 일
[나] 예시 코드도 짧게 줘.
[AI] What is the simplest Python code?


## 3. 시스템 지시문 + 멀티턴

시스템 지시문은 챗봇의 역할과 말투를 고정하는 데 사용합니다. 같은 `chat` 안에서는 이후 메시지에도 이 설정이 유지됩니다.

In [10]:
tutor_chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=(
            "당신은 친절한 Python 튜터입니다. "
            "초보자에게 쉬운 비유와 짧은 코드로 설명하세요. "
            "답변은 3문장 이내로 제한하세요."
        ),
        max_output_tokens=180,
    ),
)

print("시스템 지시문이 있는 tutor_chat 세션을 만들었습니다.")

시스템 지시문이 있는 tutor_chat 세션을 만들었습니다.


In [11]:
message = "리스트가 뭐야?"

response = call_gemini(
    "시스템 지시문 1: 리스트 설명",
    lambda: tutor_chat.send_message(message),
)

print(f"나: {message}")
show_response("튜터", response)

[API 요청 6] 시스템 지시문 1: 리스트 설명

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 6회
  잠시 후 재시도 가능: 약 2s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
나: 리스트가 뭐야?


In [12]:
message = "딕셔너리랑 차이가 뭔데?"

response = call_gemini(
    "시스템 지시문 2: 앞 대화 이어가기",
    lambda: tutor_chat.send_message(message),
)

print(f"나: {message}")
show_response("튜터", response)

[API 요청 7] 시스템 지시문 2: 앞 대화 이어가기

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 7회
  잠시 후 재시도 가능: 약 0s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
나: 딕셔너리랑 차이가 뭔데?


## 4. 수동 기록 관리

Chat 세션을 쓰지 않고도 `history`를 직접 관리하면 멀티턴처럼 동작하게 만들 수 있습니다.

In [13]:
manual_history: list[types.Content] = []


def manual_chat_turn(user_message: str):
    manual_history.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    response = call_gemini(
        "수동 history 멀티턴",
        lambda: client.models.generate_content(
            model=MODEL,
            contents=manual_history,
            config=types.GenerateContentConfig(max_output_tokens=150),
        ),
    )

    if response is not None:
        manual_history.append(
            types.Content(role="model", parts=[types.Part(text=response.text)])
        )

    return response


print("manual_history와 manual_chat_turn()을 준비했습니다.")

manual_history와 manual_chat_turn()을 준비했습니다.


In [14]:
message = "제일 좋아하는 프로그래밍 언어를 골라줘."

response = manual_chat_turn(message)
print(f"나: {message}")
show_response("AI", response)

[API 요청 8] 수동 history 멀티턴

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 8회
  잠시 후 재시도 가능: 약 54s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
나: 제일 좋아하는 프로그래밍 언어를 골라줘.


In [15]:
message = "왜 그 언어를 골랐어?"

response = manual_chat_turn(message)
print(f"나: {message}")
show_response("AI", response)

[API 요청 9] 수동 history 멀티턴
나: 왜 그 언어를 골랐어?

AI
--
제가 가장 좋아하는 프로그래

토큰 사용량: 입력=22, 출력=5, 총=168


In [16]:
print(f"현재 기록 길이: {len(manual_history)} 메시지")
for turn in manual_history:
    text = turn.parts[0].text.strip().replace("\n", " ")
    print(f"[{turn.role}] {text[:100]}")

현재 기록 길이: 3 메시지
[user] 제일 좋아하는 프로그래밍 언어를 골라줘.
[user] 왜 그 언어를 골랐어?
[model] 제가 가장 좋아하는 프로그래


## 5. 스트리밍 응답

스트리밍은 긴 답변을 받을 때 유용합니다. 출력이 생성되는 즉시 화면에 표시됩니다.

In [17]:
stream_gemini(
    "싱글턴 스트리밍",
    lambda: client.models.generate_content_stream(
        model=MODEL,
        contents="파이썬의 장점을 50자 이내로 설명해줘.",
        config=types.GenerateContentConfig(max_output_tokens=100),
    ),
)

[API 요청 10] 싱글턴 스트리밍
AI: 
[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 10회
  잠시 후 재시도 가능: 약 46s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.


In [18]:
stream_chat = client.chats.create(model=MODEL)

response = call_gemini(
    "멀티턴 스트리밍 문맥 설정",
    lambda: stream_chat.send_message("자바스크립트에 대해 얘기해볼게."),
)

show_response("문맥 설정 응답", response, limit=160)

[API 요청 11] 멀티턴 스트리밍 문맥 설정

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 11회
  잠시 후 재시도 가능: 약 44s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.


In [19]:
stream_gemini(
    "멀티턴 스트리밍",
    lambda: stream_chat.send_message_stream(
        "방금 말한 언어의 단점을 50자 이내로 알려줘."
    ),
)

[API 요청 12] 멀티턴 스트리밍
AI: 
[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 12회
  잠시 후 재시도 가능: 약 42s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.


## 6. 컨텍스트 성장 확인

아래 실습은 같은 셀을 여러 번 실행하면서 대화가 쌓일수록 입력 토큰 수가 어떻게 바뀌는지 관찰합니다.

In [20]:
context_chat = client.chats.create(model=MODEL)
context_questions = [
    "숫자 1을 영어로?",
    "숫자 2를 영어로?",
    "숫자 3을 영어로?",
]
context_index = 0

print("준비 완료. 다음 셀을 한 번 실행할 때마다 질문 1개를 보냅니다.")

준비 완료. 다음 셀을 한 번 실행할 때마다 질문 1개를 보냅니다.


In [21]:
if context_index >= len(context_questions):
    print("준비된 질문을 모두 보냈습니다. 위 준비 셀을 다시 실행하면 초기화됩니다.")
else:
    question = context_questions[context_index]
    context_index += 1

    response = call_gemini(
        f"컨텍스트 성장 {context_index}회",
        lambda: context_chat.send_message(question),
    )

    print(f"Q{context_index}: {question}")
    show_response("A", response, limit=120)

[API 요청 13] 컨텍스트 성장 1회

[Gemini API 할당량 초과]
  모델: gemini-3.5-flash
  현재 노트북 세션 요청 수: 13회
  잠시 후 재시도 가능: 약 38s
  해결: 잠시 기다렸다가 다음 셀부터 이어서 실행하세요.
  수업 팁: 꼭 필요한 예제 셀만 실행하면 quota 소모를 줄일 수 있습니다.
Q1: 숫자 1을 영어로?
